# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y

        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))

train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
    device = '/device:GPU:0'

In [6]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x

def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [7]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=(5, 5),
            padding='same',
            activation='relu'
        )
        self.conv2 = tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=(3, 3),
            padding='same',
            activation='relu'
        )

        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(
            units=num_classes,
            activation='softmax'
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores

In [8]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [9]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):

        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

        t = 0
        for epoch in range(num_epochs):

            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:

                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [10]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2
print_every = 100

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.789344072341919, Accuracy: 14.0625, Val Loss: 2.919687032699585, Val Accuracy: 11.699999809265137
Iteration 100, Epoch 1, Loss: 2.2365663051605225, Accuracy: 28.233293533325195, Val Loss: 1.8886826038360596, Val Accuracy: 38.70000076293945
Iteration 200, Epoch 1, Loss: 2.0746524333953857, Accuracy: 32.2527961730957, Val Loss: 1.8637815713882446, Val Accuracy: 37.400001525878906
Iteration 300, Epoch 1, Loss: 1.9975826740264893, Accuracy: 34.38538360595703, Val Loss: 1.8515846729278564, Val Accuracy: 37.79999923706055
Iteration 400, Epoch 1, Loss: 1.930639386177063, Accuracy: 36.097259521484375, Val Loss: 1.6990798711776733, Val Accuracy: 43.29999923706055
Iteration 500, Epoch 1, Loss: 1.8878098726272583, Accuracy: 37.072731018066406, Val Loss: 1.6381046772003174, Val Accuracy: 42.39999771118164
Iteration 600, Epoch 1, Loss: 1.858481526374817, Accuracy: 37.85097885131836, Val Loss: 1.67491614818573, Val Accuracy: 41.900001525878906
Iteration 700, Epoch 1, Lo

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [11]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.3867270946502686, Accuracy: 6.25, Val Loss: 2.331820487976074, Val Accuracy: 9.09999942779541
Iteration 100, Epoch 1, Loss: 1.8946138620376587, Accuracy: 32.67326736450195, Val Loss: 1.6842482089996338, Val Accuracy: 42.79999923706055
Iteration 200, Epoch 1, Loss: 1.7490519285202026, Accuracy: 38.751556396484375, Val Loss: 1.475976824760437, Val Accuracy: 49.39999771118164
Iteration 300, Epoch 1, Loss: 1.6598025560379028, Accuracy: 41.777408599853516, Val Loss: 1.4633146524429321, Val Accuracy: 49.20000076293945
Iteration 400, Epoch 1, Loss: 1.5874216556549072, Accuracy: 44.2331657409668, Val Loss: 1.3725343942642212, Val Accuracy: 50.80000305175781
Iteration 500, Epoch 1, Loss: 1.5373735427856445, Accuracy: 45.83645248413086, Val Loss: 1.2994134426116943, Val Accuracy: 54.29999923706055
Iteration 600, Epoch 1, Loss: 1.5074697732925415, Accuracy: 46.81000518798828, Val Loss: 1.3004109859466553, Val Accuracy: 56.599998474121094
Iteration 700, Epoch 1, Loss:

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [12]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 2.9998769760131836, Accuracy: 9.375, Val Loss: 2.827827215194702, Val Accuracy: 12.700000762939453
Iteration 100, Epoch 1, Loss: 2.246133327484131, Accuracy: 29.006805419921875, Val Loss: 1.8858258724212646, Val Accuracy: 38.29999923706055
Iteration 200, Epoch 1, Loss: 2.0813143253326416, Accuracy: 32.75808334350586, Val Loss: 1.879447102546692, Val Accuracy: 38.70000076293945
Iteration 300, Epoch 1, Loss: 2.0066888332366943, Accuracy: 34.276371002197266, Val Loss: 1.9058270454406738, Val Accuracy: 38.20000076293945
Iteration 400, Epoch 1, Loss: 1.937861442565918, Accuracy: 35.914119720458984, Val Loss: 1.7367024421691895, Val Accuracy: 41.70000076293945
Iteration 500, Epoch 1, Loss: 1.8925442695617676, Accuracy: 36.979164123535156, Val Loss: 1.6800113916397095, Val Accuracy: 42.5
Iteration 600, Epoch 1, Loss: 1.8614178895950317, Accuracy: 38.03556442260742, Val Loss: 1.6951606273651123, Val Accuracy: 42.5
Iteration 700, Epoch 1, Loss: 1.8346617221832275, Ac

Альтернативный менее гибкий способ обучения:

In [13]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 1.8161 - sparse_categorical_accuracy: 0.3894 - val_loss: 1.6279 - val_sparse_categorical_accuracy: 0.4420
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1.6217 - sparse_categorical_accuracy: 0.4490


[1.6217459440231323, 0.4490000009536743]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [14]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(filters=32, kernel_size=(5, 5), padding='same',
                               activation='relu', input_shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(filters=16, kernel_size=(3, 3), padding='same',
                               activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units=10, activation='softmax')
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iteration 0, Epoch 1, Loss: 2.3982245922088623, Accuracy: 7.8125, Val Loss: 2.347986936569214, Val Accuracy: 9.5
Iteration 100, Epoch 1, Loss: 2.112009286880493, Accuracy: 23.004331588745117, Val Loss: 1.945803165435791, Val Accuracy: 33.89999771118164
Iteration 200, Epoch 1, Loss: 2.001516580581665, Accuracy: 28.404850006103516, Val Loss: 1.8195371627807617, Val Accuracy: 37.29999923706055
Iteration 300, Epoch 1, Loss: 1.928454875946045, Accuracy: 31.405731201171875, Val Loss: 1.7296271324157715, Val Accuracy: 41.10000228881836
Iteration 400, Epoch 1, Loss: 1.8587987422943115, Accuracy: 34.164588928222656, Val Loss: 1.6569331884384155, Val Accuracy: 42.89999771118164
Iteration 500, Epoch 1, Loss: 1.8074967861175537, Accuracy: 36.0279426574707, Val Loss: 1.5806022882461548, Val Accuracy: 44.20000076293945
Iteration 600, Epoch 1, Loss: 1.7701462507247925, Accuracy: 37.401206970214844, Val Loss: 1.5448405742645264, Val Accuracy: 46.099998474121094
Iteration 700, Epoch 1, Loss: 1.73657667

In [15]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 1.6188 - sparse_categorical_accuracy: 0.4255 - val_loss: 1.4311 - val_sparse_categorical_accuracy: 0.4720
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.4566 - sparse_categorical_accuracy: 0.4701


[1.4566422700881958, 0.4700999855995178]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [16]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [17]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.8597183227539062, Accuracy: 12.5, Val Loss: 2.9366579055786133, Val Accuracy: 12.600000381469727
Iteration 100, Epoch 1, Loss: 2.2306973934173584, Accuracy: 28.29517364501953, Val Loss: 1.9352717399597168, Val Accuracy: 36.89999771118164
Iteration 200, Epoch 1, Loss: 2.069441080093384, Accuracy: 32.361629486083984, Val Loss: 1.8242192268371582, Val Accuracy: 39.79999923706055
Iteration 300, Epoch 1, Loss: 1.9965415000915527, Accuracy: 34.13621139526367, Val Loss: 1.894323468208313, Val Accuracy: 37.20000076293945
Iteration 400, Epoch 1, Loss: 1.9276891946792603, Accuracy: 35.890743255615234, Val Loss: 1.7272690534591675, Val Accuracy: 41.900001525878906
Iteration 500, Epoch 1, Loss: 1.8820518255233765, Accuracy: 36.9947624206543, Val Loss: 1.6716713905334473, Val Accuracy: 42.39999771118164
Iteration 600, Epoch 1, Loss: 1.8520030975341797, Accuracy: 37.905574798583984, Val Loss: 1.7058488130569458, Val Accuracy: 41.60000228881836
Iteration 700, Epoch 1, Lo

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [18]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(16, (3, 3), padding='same', activation='relu')
        self.pool1 = tf.keras.layers.MaxPooling2D((2, 2))

        self.conv2 = tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu')
        self.pool2 = tf.keras.layers.MaxPooling2D((2, 2))

        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(128, activation='relu')
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x

print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 2.3758392333984375, Accuracy: 10.9375, Val Loss: 2.383999824523926, Val Accuracy: 12.100000381469727
Iteration 700, Epoch 1, Loss: 1.3599787950515747, Accuracy: 51.98823165893555, Val Loss: 1.1355434656143188, Val Accuracy: 59.20000076293945
Iteration 1400, Epoch 2, Loss: 1.0064619779586792, Accuracy: 64.88435363769531, Val Loss: 0.9954083561897278, Val Accuracy: 64.30000305175781
Iteration 2100, Epoch 3, Loss: 0.8734596967697144, Accuracy: 69.59852600097656, Val Loss: 0.991012454032898, Val Accuracy: 65.20000457763672
Iteration 2800, Epoch 4, Loss: 0.7778902053833008, Accuracy: 73.13307189941406, Val Loss: 0.970440685749054, Val Accuracy: 67.0999984741211
Iteration 3500, Epoch 5, Loss: 0.7022761702537537, Accuracy: 75.6400146484375, Val Loss: 0.9880716800689697, Val Accuracy: 66.79999542236328
Iteration 4200, Epoch 6, Loss: 0.6326505541801453, Accuracy: 78.02392578125, Val Loss: 0.9529485106468201, Val Accuracy: 69.5
Iteration 4900, Epoch 7, Loss: 0.5656020

In [19]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu')
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D((2, 2))

        self.conv2 = tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu')
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D((2, 2))

        self.conv3 = tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu')
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.pool3 = tf.keras.layers.MaxPooling2D((2, 2))

        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(256, activation='relu')
        self.dropout = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = self.pool3(x)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x

print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 4.157631874084473, Accuracy: 12.5, Val Loss: 2.331364870071411, Val Accuracy: 7.90000057220459
Iteration 700, Epoch 1, Loss: 1.5362550020217896, Accuracy: 45.62455368041992, Val Loss: 1.1653239727020264, Val Accuracy: 60.39999771118164
Iteration 1400, Epoch 2, Loss: 1.1068192720413208, Accuracy: 61.301673889160156, Val Loss: 0.951192319393158, Val Accuracy: 67.79999542236328
Iteration 2100, Epoch 3, Loss: 0.9335461258888245, Accuracy: 67.65432739257812, Val Loss: 0.8378524780273438, Val Accuracy: 69.5999984741211
Iteration 2800, Epoch 4, Loss: 0.8126811981201172, Accuracy: 72.0707015991211, Val Loss: 0.7859197854995728, Val Accuracy: 74.4000015258789
Iteration 3500, Epoch 5, Loss: 0.7273807525634766, Accuracy: 74.76044464111328, Val Loss: 0.7625629901885986, Val Accuracy: 76.0
Iteration 4200, Epoch 6, Loss: 0.6480489373207092, Accuracy: 77.72489929199219, Val Loss: 0.6825649738311768, Val Accuracy: 78.0
Iteration 4900, Epoch 7, Loss: 0.5745104551315308, Accu

Опишите все эксперименты, результаты. Сделайте выводы.

Сначала был проведен эксперимент со следующей архитектурой: Conv2D(16) - MaxPool - Conv2D(32) - MaxPool - Dense(128) - Dense(10). Получена точность 66.9% на валидации, 87.5% на тренировочной выборке. В этом эксперименте отсутствуют какие-либо механизмы регуляризации. К пятой эпохе точность на валидационной выборке достигает своего максимума в 69%, после чего начинает снижаться, заканчивая также на 66.9%. При этом точность на обучающей выборке продолжает расти до 87.5%. Это признак переобучения, вызванного недостаточной емкостью сети.  Слишком малое количество фильтров и сверточных слоев не позволяет сети выучить достаточно разнообразные признаки. В итоге сеть запоминает обучающие примеры, но не способна обобщать полученные знания на новые данные.

Далее былаиспользована другая архитектура: Conv2D(32) - BN - MaxPool - Conv2D(64) - BN - MaxPool - Conv2D(128) - BN - MaxPool - Dense(256) - Dropout(0.5) - Dense(10). Здесь уже каждый сверточный слой сопровождается пакетной нормализацией, а после третьего слоя добавлен dropout. Результаты этого эксперимента демонстрируют лучшую динамику. Точность на валидации растет на протяжении всех десяти эпох, достигая 79%. Разрыв между обучающей и валидационной точностью составляет всего 5.6%, что свидетельствует об отсутствии переобучения. Пакетная нормализация позволила сети обучаться стабильно и быстро, а dropout предотвратил переобучение.

Таким образом, для набора данных CIFAR-10 двух сверточных слоев недостаточно для достижения высокой точности классификации. Минимально необходимая глубина - три слоя. Также пакетная нормализация критически важна для стабильного обучения и ускорения сходимости, особенно при использовании глубоких архитектур. Наконец, егуляризация с помощью dropout является обязательным компонентом для предотвращения переобучения, поскольку даже сеть с недостаточной емкостью в первом эксперименте начала переобучаться после пятой эпохи.